# nnInteractive — GPU Memory Benchmark

Measures peak VRAM on representative 3D volumes using `torch.cuda.max_memory_allocated()` 

(PyTorch-native, equivalent to TF's `get_memory_info()`).



**Size definitions** (by total_voxels, matching `inference_speed.ipynb`):

- **Small**: ≤ 128³ = 2,097,152

- **Medium**: 128³ – 192³

- **Large**: > 192³ = 7,077,888 (AutoZoom trigger)



Same representative volumes as `p_unet_memory.ipynb` for apples-to-apples comparison.


| **§1** | Find representative volumes (same selection logic) |

| **§2** | GPU memory measurement via PyTorch |

| **§3** | Results table (alongside P-UNet values) |

---

## §1 — Find Representative Volumes

In [ ]:
import sys
from pathlib import Path

notebook_dir = Path().resolve()
project_root = notebook_dir.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
from evaluation.benchmark_nninteractive.memory_utils import (
    scan_from_pkl, pick_largest, load_volume_for_memory,
)

In [ ]:
SMALL_LIMIT = 128 ** 3
LARGE_LIMIT = 192 ** 3

NPZ_PATHS = [
    project_root / 'data' / 'test_data' / 'FLARE_2022.npz',
    project_root / 'data' / 'test_data' / 'han_seg_ct.npz',
    project_root / 'data' / 'test_data' / 'han_seg_mri.npz',
    project_root / 'data' / 'test_data' / 'HCCTase_ceCT.npz',
    project_root / 'data' / 'test_data' / 'SegRap2023.npz',
    project_root / 'data' / 'test_data' / 'TotalSeg_mri.npz',
]

# Use the same pkl that inference_speed.ipynb uses — this ensures the memory
# bins match the actual evaluated population (all ROIs, not just largest-per-patient).
PKL_PATH = "results_p_unet_332_drop_only_ssf_none_ifl_ssf_ConfidenceDrop_df0.05_20260602_155547.pkl"

candidates = scan_from_pkl(PKL_PATH, NPZ_PATHS, small_limit=SMALL_LIMIT, large_limit=LARGE_LIMIT)

from collections import Counter
print(f'\nSize distribution:')
for label, count in Counter(c['size_bin'] for c in candidates).most_common():
    print(f'  {label}: {count}')

In [ ]:
rep = {}
for label in ['Small', 'Medium', 'Large']:
    best = pick_largest(candidates, label)
    if best:
        rep[label] = best
        print(f"{label}: {best['dataset_name']}/{best['pid']} roi={best['roi']} axis={best['axis']} total_voxels={best['total_voxels']:,}")

## §2 — GPU Memory Measurement (PyTorch / nnInteractive)

In [ ]:
import torch
from evaluation.benchmark_nninteractive.nninteractive_inference import NNInteractiveInference

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB')

In [ ]:
def profile_nninteractive(rep):
    """Run nnInteractive inference and return peak VRAM (PyTorch-native).

    Uses torch.cuda.max_memory_allocated() which reports the peak memory
    used by PyTorch ops (equivalent to TF's get_memory_info()['peak']).
    Warm-up pass first to absorb one-shot CUDA allocations.
    """
    img_3d, seg_3d_binary, prompt_2d, prompt_idx = load_volume_for_memory(rep)

    # Build initial_prompt_3d: 3D binary mask with the prompt slice set
    initial_prompt_3d = np.zeros_like(img_3d, dtype=np.float32)
    idx = [slice(None)] * 3
    idx[rep['axis']] = prompt_idx
    initial_prompt_3d[tuple(idx)] = prompt_2d

    # nnInteractive expects (1, X, Y, Z) with leading channel dim
    img_4d = np.expand_dims(img_3d, axis=0)

    # Warm-up
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(DEVICE)
    infer = NNInteractiveInference()
    _ = infer.run(
        img_4d=img_4d,
        seg_3d=seg_3d_binary,
        initial_prompt_3d=initial_prompt_3d,
        user_interacts_idx=[],
        prompt_axis=rep['axis'],
        prompt_idx=prompt_idx,
    )

    # Measure
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(DEVICE)
        torch.cuda.empty_cache()
    _ = infer.run(
        img_4d=img_4d,
        seg_3d=seg_3d_binary,
        initial_prompt_3d=initial_prompt_3d,
        user_interacts_idx=[],
        prompt_axis=rep['axis'],
        prompt_idx=prompt_idx,
    )

    peak_mb = torch.cuda.max_memory_allocated(DEVICE) / (1024 * 1024)
    return peak_mb

In [ ]:
results = {}
for label in ['Small', 'Medium', 'Large']:
    if label not in rep:
        continue
    r = rep[label]
    print(f'\\nProfiling {label} volume: {r[\"pid\"]} (roi={r[\"roi\"]}, axis={r[\"axis\"]})')
    peak_mb = profile_nninteractive(r)
    results[label] = peak_mb
    print(f'  Peak VRAM (PyTorch ops): {peak_mb:.1f} MB')

---

## §3 — Results

In [ ]:
P_UNET_MEM = {'Small': 248.2, 'Large': 362.4}  # from p_unet_memory.ipynb

print(f\"{'='*72}\")
print(f\"  GPU Memory Comparison (same RTX A6000, same volumes)\")
print(f\"  Small  <= {SMALL_LIMIT:,}   Medium  {SMALL_LIMIT:,} - {LARGE_LIMIT:,}   Large  > {LARGE_LIMIT:,} (AutoZoom)\")
print(f\"{'='*72}\")
print(f\"  {'Size':<8} {'Volume':<28} {'total_voxels':<16} {'P-UNet':<14} {'nnInteractive':<14}\")
print(f\"  {'-'*70}\")
for label in ['Small', 'Medium', 'Large']:
    if label not in rep or label not in results:
        continue
    r = rep[label]
    name = f\"{r['dataset_name']}/{r['pid']}\"
    p_mem = P_UNET_MEM.get(label, '—')
    n_mem = results[label]
    p_str = f'{p_mem:.1f} MB' if isinstance(p_mem, float) else str(p_mem)
    n_str = f'{n_mem:.1f} MB'
    print(f\"  {label:<8} {name:<28} {r['total_voxels']:<16,} {p_str:<14} {n_str:<14}\")
print(f\"{'='*72}\")
print(f\"\\n  P-UNet measured via tf.config.experimental.get_memory_info()['peak']\")
print(f\"  nnInteractive measured via torch.cuda.max_memory_allocated()\")
print(f\"  Both measured on RTX A6000 (48 GB). Same framework overhead caveat:\")
print(f\"  TF and PyTorch allocators differ; compare orders of magnitude, not bytes.\")
print(f\"\\n  Reference (nnInteractive paper, RTX 4090):\")
print(f\"    Small (<=192^3) : < 6 GB\")
print(f\"    Large (>192^3)  : < 10 GB\")
print(f\"{'='*72}\")